# G1 Academy Bonus - Task 5: robot state, services & mode switching (using the wrapper)


## Introduction
Starting today you use the finished `sdk_wrapper.G1` wrapper directly, the same way Task 1 did -- you are no longer rebuilding native `LocoClient`/`RobotStateClient`/DDS calls by hand. This task covers reading robot state, listing/toggling services, and switching FSM modes safely.

**Using Codex/AI for this task:** every method below is a finished, documented method on `sdk_wrapper.G1` -- you are not reconstructing it from DDS. If you get stuck, paste the method's docstring/signature (or the relevant line from `wrapper_cheatsheet.html`) into Codex and ask it to write the cell for you, then read what it produced before you run it against the robot. Knowing *what a call does* and *when it is safe to make it* is the point of this task, not typing it from memory.


In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)

# robot_state's RPC timeout defaults to 2s and often times out (code 3104) on service
# calls; raise it once, as academy/visualizations/service_view.py does.
g1._robot_state_client().SetTimeout(10.0)

## Task 1 - Read robot state: `get_lowstate()`, `get_odom()`, `get_battery()`, `get_state()`
- `g1.get_lowstate()` -- a snapshot dict of every motor (`q`, `dq`, `tau_est`, ...) and the IMU, taken from the latest `LowState_` message.
- `g1.get_odom()` -- the robot's current pose/twist from the odometry topic.
- `g1.get_battery()` -- a dict with the battery percentage and related BMS fields.
- `g1.get_state()` -- the composite view: FSM `id`, `mode`, `motion_mode`, `gait`, plus the battery, lowstate, service list, and SLAM info in one call. This is what to reach for when you just want "what mode is the robot in right now" -- there is no separate `get_mode()`; `get_state()["mode"]` / `get_state()["id"]` is it.


In [ ]:
# TODO: call g1.get_lowstate(), g1.get_odom(), g1.get_battery(), and g1.get_state().
raise NotImplementedError("Complete this task section")

## Task 2 - Services, and a live demo that `vui_service` gates audio
`vui_service` is the robot's **"Audio and Lighting Control Service"** -- `say()` and `set_headlight()` both go through it. Turn it **off**, call `say()` (you will hear nothing), then turn it **back on** and `say()` again (audio works). `set_service(name, enabled)` sets an explicit on/off; `get_service(name)` reads a service's status (`0` = ON, non-zero = OFF).

The setup cell already raised the `robot_state` RPC timeout, so these service calls don't fail with code `3104`.

In [ ]:
# TODO: turn vui_service off with g1.set_service("vui_service", False), call g1.say("...")
# (you should hear nothing), then turn it back on and g1.say("...") again.
raise NotImplementedError("Complete this task section")

## Task 3 - Switch modes safely: `damp_mode()` / `prepare_mode()` / `walk_mode()` / `run_mode()` / `toggle_dev_mode()`
- `damp_mode()` -- FSM `1`: bounded joint damping, no locomotion. The always-available safe fallback -- call it before an emergency stop or before releasing controller ownership.
- `prepare_mode()` -- FSM `4`: the stand-up/ready pose, the usual step before `walk_mode()`.
- `walk_mode()` -- FSM `500` on this hardware (not `501` -- see the FSM note on the slide: this academy's units run with the waist locked, only `WaistYaw` free).
- `run_mode()` -- FSM `802`.
- `toggle_dev_mode()` -- flips the `ai_sport` service; several Day 3 low-level control calls need dev mode enabled first.
Always go `damp_mode()` → `prepare_mode()` → `walk_mode()` in that order, one step at a time, checking `get_state()` between steps -- never jump straight to `run_mode()`.


In [ ]:
# TODO: g1.damp_mode(), then g1.prepare_mode(), then g1.walk_mode(), then g1.toggle_dev_mode().
raise NotImplementedError("Complete this task section")

## Final part - build a mode-switching dashboard with Codex
You have now driven every FSM transition by hand. The last exercise is to wrap those same `g1` calls in a small web dashboard -- **not** by writing it yourself, but by prompting Codex to build it against `sdk_wrapper.G1`.

A finished reference already runs on this robot: **`academy/visualizations/mode_control.py`, the mode-control dashboard served on port `8051`**. Open it in your browser first to see the target: one button per mode, a live state line, and buttons that grey out when a transition would be unsafe. Your job is to have Codex produce your own version on a *different* port so the two don't clash.

**Paste this prompt into Codex** (adjust to taste), then read the generated file before you run it against the robot:

> Using the `G1` class in `academy/sdk_wrapper.py`, write a Dash + dash-bootstrap-components web app `mode_switch_dashboard.py` that mirrors `academy/visualizations/mode_control.py` (the mode-control dashboard already running on port 8051). Requirements:
> - Construct a single `G1(iface="eth0", domain_id=0)` instance at startup and reuse it for every callback, guarded by a `threading.Lock`; never build more than one G1 per process.
> - One large button per FSM mode -- Zero Torque, Damp, Prepare, Walk, Run -- each calling the matching wrapper method: `g1.zero_torque_mode()`, `g1.damp_mode()`, `g1.prepare_mode()`, `g1.walk_mode()`, `g1.run_mode()`. Add a red **Stop** button wired to `g1.loco_stop()`.
> - A once-per-second `dcc.Interval` callback that reads `g1.get_state()` and shows the live `id` / `mode` / `motion_mode` / `gait` and battery soc in a status line, with the raw dict in a `<pre>` block.
> - Enforce the safe mode ladder: always allow Damp; disable Walk/Run unless the current `mode` is already prepare/walk/run. Wrap every robot call in try/except so one failed RPC never kills the app, and show the error text in the status line.
> - Bind to host `0.0.0.0` on port **8052** (a new port, so it does not clash with the 8051 dashboard) and print the URL on startup.

Use the cell below to launch or iterate on whatever Codex gives you.


In [ ]:
# Open-ended: run or refine the Codex-generated mode_switch_dashboard.py here.
# e.g. from a terminal:  python3 mode_switch_dashboard.py --port 8052
# then open http://<robot-ip>:8052 in your browser.


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
